# batchnorm-running-stats — ex2: train/eval BN forward: train uses batch stats; eval uses running stats — and outputs differ

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `batchnorm-running-stats`. Running the final beacon cell reports progress against the `CNN: BatchNorm running stats` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: BatchNorm running stats` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`batchnorm-running-stats`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "batchnorm-running-stats"
DD_SUBTOPIC = "CNN: BatchNorm running stats"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BatchNorm `.eval()` uses running stats, not batch stats

Ex1 updated `running_mean` and `running_var` via the EMA in train mode only. The deepening move toggles `.train()` vs `.eval()` on the same input and observes that the OUTPUT itself changes — eval mode uses the FIXED running stats, while train mode uses the batch's own mean/var.

```
train mode:  y = (x − batch_mean) / √(batch_var + ε) · γ + β
             [running_mean, running_var update via EMA]
eval  mode:  y = (x − running_mean) / √(running_var + ε) · γ + β
             [no update to running stats]
```

**Why this matters at inference.** A model trained with BN sees batch stats during training; at deployment the batch may be size 1 (single image), and batch stats become meaningless or undefined (var = 0 for B=1). Eval mode swaps to the running stats accumulated over training, giving stable, batch-independent outputs.

**Same module, two outputs.** Calling `bn.eval()` then `bn.train()` on the same `bn` and feeding the same `x` returns DIFFERENT tensors. The difference is the canonical 'why does my model behave weirdly at inference' debugging step for any new BN user.

### Exercise 2 — train/eval BN forward: train uses batch stats; eval uses running stats — and outputs differ

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyse the BatchNorm forward pass under `.train()` vs `.eval()`: train mode normalises by the BATCH's mean/var (and updates the running stats), while eval mode normalises by the FIXED running stats — so the same `x` through the same `bn` produces DIFFERENT outputs in the two modes.
> Keywords: batchnorm, train, eval, running-stats
> ```

**KCs targeted:** `train-mode-uses-batch-mean-var`, `eval-mode-uses-running-mean-var`

Implement `ex2_bn_train_vs_eval_forward(x, bn)`. Same `bn` module, same input `x`, run forward in both modes.

Steps:
1. `bn.train()` → call `y_train = bn(x)`. This uses the batch's OWN mean/var AND updates `bn.running_mean` / `bn.running_var` via the EMA (PyTorch handles this internally).
2. `bn.eval()` → call `y_eval = bn(x)`. This uses the (now-updated) `running_mean` / `running_var` and does NOT update them.
3. Return tuple `(y_train, y_eval)`.

Do NOT manually rebuild the BN formula — call `bn(x)` and let PyTorch swap the stats based on the mode. The drill is about OBSERVING the mode-dependent behaviour, not re-deriving it.

Inputs:
- `x`: `(B, C, H, W)` for `BatchNorm2d`, or `(B, C)` for `BatchNorm1d`.
- `bn`: an `nn.BatchNorm2d` or `nn.BatchNorm1d` instance with pre-existing `running_mean` / `running_var` (defaults are 0 / 1).

Output: `(y_train, y_eval)` — both Tensors of shape `x.shape`.

In [ ]:
def ex2_bn_train_vs_eval_forward(x, bn):
    bn.train()
    y_train = bn(x)
    bn.eval()
    y_eval = bn(x)
    return y_train, y_eval


<details><summary>Solution</summary>

```python
def ex2_bn_train_vs_eval_forward(x, bn):
    bn.train()
    y_train = bn(x)
    bn.eval()
    y_eval = bn(x)
    return y_train, y_eval
```

**Mode toggling is the whole API.** PyTorch's BN does the stat-swap internally based on `self.training`. The Module-level `.train()` / `.eval()` flip flips this flag on every child (including BNs) — no need to thread a `training=True` arg through.

**Train pass UPDATES running stats; eval pass does NOT.** The second test confirms this by capturing `running_mean`/`var` after a train pass, then running multiple eval passes and checking the stats haven't moved.

**Why this is the canonical deploy bug.** A user who forgets to call `model.eval()` before inference gets the BN-in-train-mode behaviour: each inference batch's own stats are used. For a single-image inference call, batch_var ≈ 0 → division by ε → garbled outputs. The two-line `model.eval()` fix is invisible from the loss curves during training.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()